In [4]:
from google.colab import drive
drive.mount('/content/drive')   # volg de autorisatieprompt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Build 'patched_notebook.ipynb' using nbformat (Option B)
# - Writes to Drive if mounted at /content/drive, else to /content
# - Includes all 9 cells (markdown + code) and sets cell IDs per nbformat 4.5+
# Refs: nbformat API & JEP 62 (cell ids)

import os, nbformat as nbf
from uuid import uuid4

def md(text: str):
    c = nbf.v4.new_markdown_cell(text)
    c["id"] = str(uuid4())
    return c

def code(text: str):
    c = nbf.v4.new_code_cell(text, execution_count=None, outputs=[])
    c["id"] = str(uuid4())
    return c

# Choose output path
OUT_DIR = "/content/drive/MyDrive/test_thesis/test_thesis_aoyang/data_labeled/data_multiplexed_gtscreen/" if os.path.exists("/content/drive/MyDrive/test_thesis/test_thesis_aoyang/data_labeled/data_multiplexed_gtscreen/") else "/content"
OUT_PATH = os.path.join(OUT_DIR, "2_modeling_classification.ipynb")

nb = nbf.v4.new_notebook()
nb["nbformat"] = 4
nb["nbformat_minor"] = 5
nb["metadata"].update({
    "kernelspec": {"name": "python3", "display_name": "Python 3"},
    "language_info": {"name": "python"},
})

cells = []

# 1) Markdown — Modelkeuze
cells.append(md("Modelkeuze: GBDT (conform ESP)\n"))

# 2) Markdown — Preflight results
cells.append(md("""## Preflight results
- Patch 1 (HF call-signature): **Applied** — vervangen `mdl(*enc)` → `mdl(**enc)` en assert op hidden size 1280.
- Patch 2 (glob-pattern aggregatie): **Applied** — gebruikt `glob(str(mdir/'fold*.json'))`.
- Patch 3 (RDKit distributie in Colab): **Applied** — pin `rdkit-pypi==2023.9.5` i.p.v. `rdkit`.
- Patch 4 (Duplicaat-statements): **Skipped** — geen functionele duplicaten aangetroffen die state overschrijven.
- Patch 5 (ConvertToNumpyArray buffer): **Applied** — 1D buffer `np.zeros(NBITS, np.uint8)` + toewijzing.
- Patch 6 (Label-guard cs_double): **Applied** — gebruikt `df.get('cs_double','')` en idem voor `cs_single`.
- Patch 7 (Pins & runtime-herstart): **Applied** — exacte versiepins + comment voor runtime-restart.
- Patch 8 (Determinisme & seeds): **Applied** — seeds voor `random/numpy/torch` + `torch.use_deterministic_algorithms(True)` en cuDNN-flags.
- Patch 9 (HF model-revision pin): **Applied** — `revision='7b37824baec4d3658e1df7479222a7c79b465b76'` toegevoegd en assert embed-dim=1280.
- Patch 10 (ESM batching & OOM fallback): **Applied** — try/except op CUDA OOM → batch halveren / CPU fallback.
- Patch 11 (Zelfvoorzienende build-cel): **Applied** — één cel bouwt TSV→FPs/ESM→indices en scaffold-splits indien ontbreken.
- Patch 12 (Padstructuur & persistentie): **Applied** — `PROJ/{data,features,splits,models,metrics,logs,reports}` + Drive-mount optie.
- Patch 13 (Artefact- en aggregatiepaden): **Applied** — consistente namen en paden conform contract.
- Patch 14 (Notebook JSON-vereisten): **Applied** — nbformat v4.5, unieke cel-IDs, `execution_count=null`, `outputs=[]`.
"""))

# 3) Code — Environment & pins
cells.append(code("""# Environment & pins
# NOTE: If packages are newly installed, use: Runtime > Restart runtime, then re-run all cells.
!pip install --quiet \\
  numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2 \\
  xgboost==1.7.6 \\
  rdkit-pypi==2023.9.5 \\
  transformers==4.44.2 tokenizers==0.19.1 huggingface_hub==0.24.6 \\
  torch==2.3.1 \\
  gdown==5.2.0

print("If packages were freshly installed, restart the runtime and re-run the notebook.")
"""))

# 4) Code — Init & seeds
cells.append(code("""# Init & seeds
import os, json, random, logging
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception as e:
    print(f"Determinism setup note: {e}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU available:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU (GBDT trains on CPU; ESM inference will fallback to CPU).')

N_JOBS = os.cpu_count() or 2
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
"""))

# 5) Code — Project root & folders
cells.append(code("""# Project root (Drive of lokaal) & mappen
USE_DRIVE = True  # zet op False voor volledige herbouw per run zonder persistentie
PROJ_NAME = 'esi_baseline_gbdt'
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        PROJ = Path('/content/drive/MyDrive')/PROJ_NAME
    except Exception as e:
        print('Drive mount failed; falling back to local /content (ephemeral).', e)
        PROJ = Path('/content')/PROJ_NAME
else:
    PROJ = Path('/content')/PROJ_NAME

for sub in ['data','features','splits','models','metrics','logs','reports']:
    (PROJ/sub).mkdir(parents=True, exist_ok=True)
print('PROJ =', PROJ)
"""))

# 6) Code — Build artifacts (TSV -> ESM/Morgan -> splits)
cells.append(code(r"""# Build artifacts (2.1–2.4): reactions.tsv, ESM-1b embeddings, Morgan FPs, scaffold-splits
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from transformers import AutoTokenizer, AutoModel
import gdown

DATA_TSV = PROJ/'data'/'reactions.tsv'
FEAT_DIR = PROJ/'features'
SPLITS_JSON = PROJ/'splits'/'scaffold_folds.json'

# ---- 2.1 Download & Load Reaction TSV ----
if not DATA_TSV.exists():
    file_id = os.environ.get('REACTIONS_TSV_GDRIVE_ID', '').strip()
    if file_id:
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, str(DATA_TSV), quiet=False)
    else:
        # Minimal fallback: instruct user to upload; keeps notebook self-voorzienend in Colab UI
        raise FileNotFoundError('Missing PROJ/data/reactions.tsv. Set env REACTIONS_TSV_GDRIVE_ID or upload reactions.tsv to PROJ/data/.')

df = pd.read_csv(DATA_TSV, sep='\t' if DATA_TSV.suffix=='.tsv' else ',', dtype=str).fillna('')
required_cols = ['enzyme','sequence','csmiles']
for c in required_cols:
    assert c in df.columns, f"Missing required column: {c}"
# label: reaction = 1 if cs_single or cs_double present; otherwise 0
if 'reaction' not in df.columns:
    cs_single = df.get('cs_single','')
    cs_double = df.get('cs_double','')
    df['reaction'] = ((cs_single!='') | (cs_double!='')).astype(int)

# Canonicalize SMILES
def canon_smiles(smi: str) -> str:
    try:
        m = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(m, canonical=True) if m else ''
    except Exception:
        return ''

df['csmiles'] = df['csmiles'].apply(canon_smiles)
df = df[df['csmiles']!=''].reset_index(drop=True)

# ---- 2.3 Generate Morgan fingerprints ----
RADIUS = 2
NBITS = 1024
subs = sorted(df['csmiles'].unique())
sub_to_idx = {s:i for i,s in enumerate(subs)}
fps = np.zeros((len(subs), NBITS), dtype=np.uint8)
bad = 0
for s,i in sub_to_idx.items():
    m = Chem.MolFromSmiles(s)
    if m is None:
        bad += 1
        continue
    bv = AllChem.GetMorganFingerprintAsBitVect(m, RADIUS, nBits=NBITS, useChirality=False, useBondTypes=True)
    arr = np.zeros(NBITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bv, arr)
    fps[i] = arr
np.save(FEAT_DIR/'morgan_fp.npy', fps)
pd.DataFrame({'sub_idx': list(range(len(subs))), 'smiles': subs}).to_csv(FEAT_DIR/'sub_index.csv', index=False)
print(f"Morgan FPs: {fps.shape}, invalid smiles: {bad}")

# ---- 2.2 & 2.4 Run Inference for ESM-1b embeddings ----
enz = sorted(df['sequence'].unique())
enz_to_idx = {s:i for i,s in enumerate(enz)}
EMB_NPY = FEAT_DIR/'esm1b_emb.npy'
if not EMB_NPY.exists():
    MODEL_NAME = 'facebook/esm1b_t33_650M_UR50S'
    HF_REV = os.environ.get('ESM1B_REV', '7b37824baec4d3658e1df7479222a7c79b465b76')  # pinned
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=HF_REV)
    mdl = AutoModel.from_pretrained(MODEL_NAME, revision=HF_REV).to(DEVICE)
    mdl.eval()
    BATCH_SIZE = 8
    embs = np.zeros((len(enz), 1280), dtype=np.float32)
    i = 0
    local_device = DEVICE
    with torch.no_grad():
        while i < len(enz):
            bs = min(BATCH_SIZE, len(enz) - i)
            while True:
                try:
                    batch = enz[i:i+bs]
                    enc = tok(batch, return_tensors='pt', padding=True, truncation=True)
                    enc = {k: v.to(local_device) for k,v in enc.items()}
                    out = mdl(**enc).last_hidden_state  # [B, L, 1280]
                    assert out.size(-1) == 1280, f"Unexpected hidden size: {out.size(-1)}"
                    mask = enc['attention_mask'].unsqueeze(-1)
                    pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                    embs[i:i+bs, :] = pooled.detach().cpu().numpy()
                    i += bs
                    break
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower() and local_device == 'cuda':
                        torch.cuda.empty_cache()
                        if bs > 1:
                            bs = max(1, bs//2)
                            print(f"[ESM] CUDA OOM — reducing batch size to {bs}")
                            continue
                        else:
                            print('[ESM] CUDA OOM at batch size 1 — falling back to CPU')
                            mdl = mdl.to('cpu')
                            local_device = 'cpu'
                            continue
                    raise
    np.save(EMB_NPY, embs)
pd.DataFrame({'enz_idx': list(range(len(enz))), 'sequence': enz}).to_csv(FEAT_DIR/'enz_index.csv', index=False)
print('ESM embeddings ready:', np.load(EMB_NPY).shape)

# ---- Generate scaffold-based 5-fold CV if missing ----
if not SPLITS_JSON.exists():
    # Build pair index
    df_pairs = df[['sequence','csmiles','reaction']].copy()
    df_pairs['enz_idx'] = df_pairs['sequence'].map(enz_to_idx)
    df_pairs['sub_idx'] = df_pairs['csmiles'].map(sub_to_idx)
    def scaffold(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return ''
        scaf = MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(scaf) if scaf else ''
    df_pairs['scaffold'] = df_pairs['csmiles'].apply(scaffold)

    # Assign scaffolds to folds round-robin, stratified by label frequency per scaffold
    scaf_groups = df_pairs.groupby('scaffold')
    scafs = list(scaf_groups.groups.keys())
    scafs.sort(key=lambda s: -scaf_groups.get_group(s)['reaction'].sum() if s in scaf_groups.groups else 0)
    K = 5
    folds = {k: {'train': [], 'valid': []} for k in range(K)}
    scaf_to_fold = {s: (i % K) for i, s in enumerate(scafs)}
    for idx, row in df_pairs.iterrows():
        k = scaf_to_fold.get(row['scaffold'], idx % K)
        folds[k]['valid'].append(int(idx))
    all_ids = set(range(len(df_pairs)))
    for k in range(K):
        val_ids = set(folds[k]['valid'])
        folds[k]['train'] = sorted(list(all_ids - val_ids))
        folds[k]['valid'] = sorted(list(val_ids))
    with open(SPLITS_JSON, 'w') as f:
        json.dump(folds, f)
    print('Scaffold CV created:', SPLITS_JSON)
else:
    print('Using existing splits:', SPLITS_JSON)
"""))

# 7) Code — CV training (GBDT)
cells.append(code(r"""# CV training (GBDT) met grid: leest splits; schrijft metrics/models
from itertools import product
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, brier_score_loss
from xgboost import XGBClassifier

DF = pd.read_csv(PROJ/'data'/'reactions.tsv', sep='\t' if (PROJ/'data'/'reactions.tsv').suffix=='.tsv' else ',').fillna('')
if 'reaction' not in DF.columns:
    DF['reaction'] = ((DF.get('cs_single','')!='') | (DF.get('cs_double','')!='')).astype(int)
DF = DF[DF['csmiles']!=''].reset_index(drop=True)

enz_index = pd.read_csv(PROJ/'features'/'enz_index.csv')
sub_index = pd.read_csv(PROJ/'features'/'sub_index.csv')
enz_to_idx = dict(zip(enz_index['sequence'], enz_index['enz_idx']))
sub_to_idx = dict(zip(sub_index['smiles'], sub_index['sub_idx']))
embs = np.load(PROJ/'features'/'esm1b_emb.npy')
fps  = np.load(PROJ/'features'/'morgan_fp.npy')

pairs = DF[['sequence','csmiles','reaction']].copy()
pairs['enz_idx'] = pairs['sequence'].map(enz_to_idx)
pairs['sub_idx'] = pairs['csmiles'].map(sub_to_idx)
pairs = pairs.dropna(subset=['enz_idx','sub_idx']).reset_index(drop=True)

with open(PROJ/'splits'/'scaffold_folds.json','r') as f:
    folds = json.load(f)

grid = {
  'learning_rate': [0.01, 0.05],
  'max_depth': [6, 10],
  'n_estimators': [400, 800],
  'subsample': [0.7, 1.0]
}
grid_list = list(product(grid['learning_rate'], grid['max_depth'], grid['n_estimators'], grid['subsample']))

for k_str, fold in folds.items():
    k = int(k_str)
    fold_dir = PROJ/'models'/'gbdt'/f'fold{k}'
    (fold_dir).mkdir(parents=True, exist_ok=True)
    mdir = PROJ/'metrics'/'cv'/'gbdt'
    mdir.mkdir(parents=True, exist_ok=True)

    tr_ids = fold['train']
    va_ids = fold['valid']
    tr = pairs.iloc[tr_ids]
    va = pairs.iloc[va_ids]

    X_tr = np.hstack([embs[tr['enz_idx'].astype(int)], fps[tr['sub_idx'].astype(int)]])
    y_tr = tr['reaction'].astype(int).values
    X_va = np.hstack([embs[va['enz_idx'].astype(int)], fps[va['sub_idx'].astype(int)]])
    y_va = va['reaction'].astype(int).values

    # handle imbalance
    pos = (y_tr==1).sum()
    neg = (y_tr==0).sum()
    scale_pos = max(1.0, neg / max(1, pos))

    best = {'roc_auc': -1, 'params': None, 'model': None}
    for lr, md, ne, ss in grid_list:
        clf = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            learning_rate=lr, max_depth=md, n_estimators=ne, subsample=ss,
            scale_pos_weight=scale_pos, n_jobs=N_JOBS, random_state=SEED
        )
        clf.fit(X_tr, y_tr)
        p_va = clf.predict_proba(X_va)[:,1]
        yhat = (p_va >= 0.5).astype(int)
        roc = roc_auc_score(y_va, p_va)
        acc = accuracy_score(y_va, yhat)
        mcc = matthews_corrcoef(y_va, yhat)
        bri = brier_score_loss(y_va, p_va)
        if roc > best['roc_auc']:
            best = {
                'roc_auc': roc, 'acc': acc, 'mcc': mcc, 'brier': bri,
                'params': {'learning_rate': lr, 'max_depth': md, 'n_estimators': ne, 'subsample': ss},
                'model': clf
            }
    # save best model and metrics
    best['model'].get_booster().save_model(str(fold_dir/'model.bin'))
    with open(mdir/f'fold{k}.json','w') as f:
        out = {'fold': k, 'metrics': {k2: float(v) for k2,v in best.items() if k2 in ['roc_auc','acc','mcc','brier']}, 'best_params': best['params']}
        json.dump(out, f, indent=2)
    print(f"Fold {k} | ROC-AUC={best['roc_auc']:.3f} ACC={best['acc']:.3f} MCC={best['mcc']:.3f}")
"""))

# 8) Code — Aggregatie & rapportage
cells.append(code(r"""# Aggregatie & rapportage
from glob import glob

mdir = PROJ/'metrics'/'cv'/'gbdt'
fold_files = sorted(glob(str(mdir/'fold*.json')))
rows = []
for fp in fold_files:
    with open(fp,'r') as f:
        rows.append(json.load(f))
if rows:
    dfm = pd.json_normalize(rows)
    df_sum = pd.DataFrame({
        'metric': ['roc_auc','acc','mcc','brier'],
        'mean': [dfm['metrics.roc_auc'].mean(), dfm['metrics.acc'].mean(), dfm['metrics.mcc'].mean(), dfm['metrics.brier'].mean()],
        'std':  [dfm['metrics.roc_auc'].std(), dfm['metrics.acc'].std(), dfm['metrics.mcc'].std(), dfm['metrics.brier'].std()],
    })
    df_sum.to_csv(PROJ/'metrics'/'cv'/'gbdt'/'summary.csv', index=False)
    # simple markdown report
    grid_line = 'learning_rate∈{0.01,0.05}; max_depth∈{6,10}; n_estimators∈{400,800}; subsample∈{0.7,1.0}'
    report = [
        '# Baseline CV Report — GBDT (conform ESP)\n',
        f'- Splits: {PROJ}/splits/scaffold_folds.json\n',
        f'- Hyperparameter grid: {grid_line}\n',
        '\n',
        '## Summary metrics (mean ± std)\n'
    ]
    for _,r in df_sum.iterrows():
        report.append(f"- {r['metric']}: {r['mean']:.4f} ± {r['std']:.4f}\n")
    (PROJ/'reports'/'baseline_cv_report.md').write_text(''.join(report))
    print('Wrote summary and report.')
else:
    print('No fold metrics found to aggregate.')
"""))

# 9) Markdown — Deliverables
cells.append(md("""## Deliverables

| WBS-stap | Afhankelijke stap(pen) | Notebookbron | Uit te voeren taak | Verwachte output |
|---|---|---|---|---|
| 5.1 | 1. Initialization; 2.1–2.4 | 1.1, 1.2, 2.1–2.4 | Zet seeds (SEED=42; random/numpy/torch) en deterministische flags; device-detectie (GPU voor ESM inference indien beschikbaar; GBDT op CPU); kies persistentie: Drive-mount onder `PROJ` of herbouw per run. | Logregels; mapstructuur `PROJ/{data,features,splits,models,metrics,logs,reports}` |
| 5.1.1 | 2.1, 2.2, 2.3, 2.4 | 2.1, 2.2, 2.3, 2.4 | Voorbereid trainingsscripts: laad `features/esm1b_emb.npy` + `features/enz_index.csv` en `features/morgan_fp.npy` + `features/sub_index.csv`; join op `(sequence,csmiles)`; logs naar `logs/`. | `features/esm1b_emb.npy`, `features/enz_index.csv`, `features/morgan_fp.npy`, `features/sub_index.csv` |
| 5.1.2 | 5.1.2.1; 5.1.2.2 | 2.4 (splits) | Cross-validation over scaffold-splits; splits aanwezig als `splits/scaffold_folds.json` (fold→{train,valid}); indien afwezig: **N/A + minimale toevoeging** (genereer scaffold-splits). | `models/gbdt/fold{K}/model.bin`, `metrics/cv/gbdt/fold{K}.json` |
| 5.1.2.1 | 2.4 | 2.4 | Schrijf CV-trainingsscript met hyperparametergrid **GBDT**: “learning_rate∈{0.01,0.05}; max_depth∈{6,10}; n_estimators∈{400,800}; subsample∈{0.7,1.0}”; gebruik `scale_pos_weight`. | Per fold beste params in `metrics/cv/gbdt/fold{K}.json` |
| 5.1.2.2 | 5.1.2.1 | N/A | Voer CV-runs lokaal (sequentieel/parallel via `n_jobs`); GBDT training op CPU (`tree_method='hist'`); ESM inference op GPU indien beschikbaar. | `models/gbdt/fold{K}/model.bin`, logs |
| 5.1.2.3 | 5.1.2 | N/A | Agregeer foldresultaten en bereken ROC-AUC, ACC, MCC, Brier; schrijf samenvatting (mean±std). | `metrics/cv/gbdt/summary.csv` |
| 5.1.2.4 | 5.1.2.3 | N/A | Documenteer CV-proces en uitkomsten (grid, splits, metrics) in Markdown. | `reports/baseline_cv_report.md` |
| 5.1.3 | 5.1.2 | N/A | Registreer prestaties en bewaar getrainde artefacten per fold; vaste paden onder `models/gbdt/fold{K}/` en `metrics/cv/gbdt/`. | `models/gbdt/fold{K}/model.bin`, `metrics/cv/gbdt/fold{K}.json` |

### Checklist (max. 8 stappen)
1. Mount Drive of kies herbouw per run en maak `PROJ`-mappen aan.
2. Pin dependencies en **restart runtime** indien nodig.
3. Init seeds/determinisme en device-detectie (GPU alleen voor ESM inference).
4. Bouw/valideer artifacts: `reactions.tsv` → `features/esm1b_emb.npy` & `features/morgan_fp.npy` (+ index CSV’s).
5. Genereer of laad `splits/scaffold_folds.json` (fold→{train,valid}).
6. Train **GBDT** per fold over grid en bewaar `model.bin` + fold-metrics JSON.
7. Agregeer metrics en schrijf `metrics/cv/gbdt/summary.csv`.
8. Genereer `reports/baseline_cv_report.md` met grid, splits en samenvatting.
"""))

# (Optional) Markdown — Citations (kept as plain links)
cells.append(md("""## Citations
- nbformat API (read/write/new cells): https://nbformat.readthedocs.io/en/latest/api.html
- Jupyter Notebook Format & cell ids (4.5+): https://nbformat.readthedocs.io/en/latest/format_description.html
- JEP 62 — Cell ID Addition: https://jupyter.org/enhancement-proposals/62-cell-id/cell-id.html
- Colab — Drive mount & I/O: https://colab.research.google.com/notebooks/io.ipynb
"""))

nb["cells"] = cells
nbf.write(nb, OUT_PATH)
print(f"Wrote: {OUT_PATH}")


Wrote: /content/drive/MyDrive/test_thesis/test_thesis_aoyang/data_labeled/data_multiplexed_gtscreen/2_modeling_classification.ipynb
